# ETL Helper Functions Demo

This notebook shows how to use the generic ETL helpers in this project:

- Load data into a `pandas.DataFrame`
- Use SQLAlchemy models and a database session
- Apply generic transformations (lookups, column removal, computed and required columns)
- Generate SQL `INSERT` statements from the final DataFrame

You can run the cells step by step after activating the virtualenv and launching Jupyter in this project directory.


In [ ]:
# Basic setup: imports and database initialization

from datetime import datetime

import pandas as pd

from db import Base, engine, get_session
from models import Customer
from transformer import (
    TransformationConfig,
    replace_column_with_lookup,
    remove_columns,
    add_computed_column,
    add_missing_columns,
)
from sql_writer import dataframe_to_insert_statements

# Create all tables defined in models.py (idempotent)
Base.metadata.create_all(bind=engine)

print("Database tables created.")


In [ ]:
# Seed a sample Customer row for lookup examples

with get_session() as session:
    # Ensure there is at least one customer with a known TIN
    existing = session.query(Customer).filter_by(tin_number="123-45-6789").first()
    if existing is None:
        c = Customer(
            tin_number="123-45-6789",
            customer_name="Acme Corp",
            customer_vat_number="VAT-ACME-001",
            email="info@acme.example",
            region="North",
            city="Gotham",
            business_type="Manufacturing",
            primary_contact="John Doe",
            secondary_contact="Jane Smith",
        )
        session.add(c)
        session.flush()  # assign ID
        print("Inserted Customer with ID:", c.id)
    else:
        print("Customer already present with ID:", existing.id)


In [ ]:
# Load example input data into a DataFrame
# Here we use the CSV example (you can also save it as XLSX and use excel_loader)

from pathlib import Path

csv_path = Path("example_data/customers_example.csv")

if not csv_path.exists():
    raise FileNotFoundError(csv_path)

input_df = pd.read_csv(csv_path)
input_df


In [ ]:
# Build a generic TransformationConfig for the customers example


def full_name_func(row):
    # Example: compute a display name from customer_name
    name = row.get("customer_name") or ""
    return str(name).strip()


config = TransformationConfig(
    replace_lookups=[
        {
            "source_column": "customer_tin",
            # Use the generic "module:ClassName" syntax so the transformer can resolve it
            "model": "models:Customer",
            "lookup_field": "tin_number",
            "return_field": "id",
            "new_column_name": "customer_id",
        }
    ],
    remove_columns=["temp_column", "unused_flag"],
    computed_columns=[
        {
            "new_column": "full_name",
            "function": full_name_func,
        }
    ],
    required_columns={
        "created_at": datetime.utcnow,
    },
)

with get_session() as session:
    transformed_df = config.apply(input_df.copy(), session)

transformed_df


In [ ]:
# Generate SQL INSERT statements from the transformed DataFrame

statements = dataframe_to_insert_statements(
    transformed_df,
    table_name="customers",
)

print("First generated statement:\n", statements[0])
print("\nTotal statements:", len(statements))
